# Setup

In [ ]:
import os
import sys
import time
import json
import urllib3
import traceback
from random import choice, seed
from tqdm import tqdm

In [ ]:
sys.path.append('..')

In [ ]:
from src.generate_persona import generate_agent_pool_pro, generate_agent_pool_con
from src.connectors import parse_request_and_execute
from src.debate import start_debate, next_debate_round, next_debate_round_toxic

In [ ]:
with open('../conf/api_key.txt', 'r') as file:
    API_KEY = file.read().strip()

In [ ]:
with open('../conf/url.txt', 'r') as file:
    url = file.read().strip()

In [ ]:
# need to disable verification of certificate, this removes the re-occuring warnings 
urllib3.disable_warnings()

In [ ]:
headers = {
    "Authorization": f"Bearer {API_KEY}"
}

In [ ]:
options_dict = {
    "temperature": "0",
    "seed": "42"
}
n_agents = 10

In [ ]:
with open(f'../data/debate_topics.json', 'r') as fp:
    topics = json.load(fp)

In [ ]:
n = 1

# Generate Personas

In [ ]:
for topic_ix, proposition in enumerate(topics.keys()):
    topic_nr = topic_ix + 1
    
    file_path = f'../data/pool_of_{n_agents}_debate_pro_agents_for_topic_{topic_nr}.json'
    #if topic_nr == 1:
    #    continue

    if os.path.isfile(file_path):
        print(f"pro-pool for topic nr. {topic_nr} already exists at {file_path} - {proposition}")
    else:
    
        print(f"processing pro-pool for topic nr. {topic_nr} - {proposition}")
    
        start_time = time.time()

        response = parse_request_and_execute(
            generate_agent_pool_pro(proposition, n_agents), 
            url = url,
            headers = headers,
            options_dict = options_dict,
            max_retries=5,
            delay=60
        )
    
        total_time = time.time() - start_time
        print(f"--- Topic {topic_nr}: {total_time / 60:.2f} minutes --- {proposition}")
        
        pool_of_pro_debate_agents = [
            json.loads(("{" + resp.split("}")[0].split(" (modified to) ")[0] + "}").strip())
            for resp in 
            response.json()["choices"][0]["message"]["content"].split("{")
            if "}" in resp
        ]
        
        with open(file_path, 'w') as fp:
            json.dump(pool_of_pro_debate_agents, fp)

    file_path = f'../data/pool_of_{n_agents}_debate_con_agents_for_topic_{topic_nr}.json'

    if os.path.isfile(file_path):
        print(f"con-pool for topic nr. {topic_nr} already exists at {file_path} - {proposition}")
    else:
    
        print(f"processing con-pool for topic nr. {topic_nr} - {proposition}")
    
        start_time = time.time()
    
        response = parse_request_and_execute(
            generate_agent_pool_con(proposition, n), 
            url, 
            headers, 
            options_dict, 
            max_retries=5,
            delay=60
        )
    
        total_time = time.time() - start_time
        print(f"--- Topic {topic_nr}: {total_time / 60:.2f} minutes --- {proposition}")
        
        pool_of_con_debate_agents = [
            json.loads(("{" + resp.split("}")[0].split(" (modified to) ")[0] + "}").strip())
            for resp in 
            response.json()["choices"][0]["message"]["content"].split("{")
            if "}" in resp
        ]
        
        with open(file_path, 'w') as fp:
            json.dump(pool_of_con_debate_agents, fp)


# Debate

In [ ]:
seed(42)
N = 1
n = 10

os.makedirs('../data/test_runs', exist_ok=True)  # GEAENDERT (Felix): Ordner automatisch anlegen

for j in range(1, N+1):
    topic_ix, proposition = choice(list(enumerate(topics.keys())))
    topic_nr = topic_ix + 1

    file_path = f'../data/pool_of_{n}_debate_con_agents_for_topic_{topic_nr}.json'
    with open(file_path, 'r') as fp:
        pool_of_con_debate_agents = json.load(fp)

    file_path = f'../data/pool_of_{n}_debate_pro_agents_for_topic_{topic_nr}.json'
    with open(file_path, 'r') as fp:
        pool_of_pro_debate_agents = json.load(fp)

    print("discussion", j, ", proposition", proposition)


    for toxicity_level in ["no", "heavy"]:

        debate_missing = True
        i = 0
        # initialize discussion
        while debate_missing:
            try:
                debate = start_debate(
                    proposition=proposition,
                    topic_ix=topic_ix,
                    topic_nr=topic_nr,
                    pool_of_con_debate_agents=pool_of_con_debate_agents,
                    n_con_agents=1,
                    pool_of_pro_debate_agents=pool_of_pro_debate_agents,
                    n_pro_agents=1,
                    url = url,
                    options_dict=options_dict,
                    headers = headers
                )
                # not missing anymore
                debate_missing = False
            except Exception as e:
                print("failed:", type(e).__name__, "–", e)
            i += 1
            if i > 100:
                break

        # preparing discussion

        pro_or_con_toxic = choice(["pro", "con"])
        print("pro_or_con_toxic", pro_or_con_toxic)

        print("toxicity_level", toxicity_level)
        debate["pro_or_con_toxic"] = pro_or_con_toxic


        if toxicity_level == "no":
            i = 0
            still_arguing = True
            # running discussion
            while still_arguing:
                try:
                    debate = next_debate_round(debate, url, headers, options_dict)
                    # GEAENDERT (Felix): Speicherordner test_runs statt toxic_and_baseline_random_yourmodel
                    file_path = f'../data/test_runs/discussion_{j}_round_{i}_for_topic_{topic_nr}_neither_toxic_level_{toxicity_level}.json'
                    with open(file_path, 'w') as fp:
                        json.dump(debate, fp)
                    if debate["in_alignment"]:
                        # then it would be over
                        still_arguing = False
                except Exception as e:
                    print("failed:", type(e).__name__, "–", e)
                    print(traceback.format_exc())
                    debate["nrounds"] =  debate["nrounds"] - 1
                i += 1
                if i > 100:
                    break
        else:
            i = 0
            still_arguing = True
            # running discussion
            while still_arguing:
                try:
                    debate = next_debate_round_toxic(debate, url, headers, options_dict, pro_or_con_toxic, persuadability=0.5, toxicity_level=toxicity_level)
                    # GEAENDERT (Felix): Speicherordner test_runs statt toxic_and_baseline_random_oss
                    file_path = f'../data/test_runs/discussion_{j}_round_{i}_for_topic_{topic_nr}_{pro_or_con_toxic}_toxic_level_{toxicity_level}.json'
                    with open(file_path, 'w') as fp:
                        json.dump(debate, fp)
                    if debate["in_alignment"]:
                        # then it would be over
                        still_arguing = False
                except Exception as e:
                    print("failed:", type(e).__name__, "–", e)
                    print(traceback.format_exc())
                    debate["nrounds"] =  debate["nrounds"] - 1
                i += 1
                if i > 100:
                    break
        print("")